# Hello, _nbpresent_!

In [1]:
import nbpresent
nbpresent.__version__

'3.0.2'

## Exploratory data analysis

In [2]:
import pandas as pd
import numpy as np
import os
import matplotlib.pylab as plt
%matplotlib inline

In [3]:
pd.set_option('display.width', 500)
pd.set_option('display.max_columns', 100)

In [4]:
user_info_train = pd.read_csv('./Credit/train/user_info_train.txt',names=['ID','gender','career','education','marriage','hukou'],header=None)
loan_time_train = pd.read_csv('./Credit/train/loan_time_train.txt',names=['ID','timestamp_money'],header=None)
overdue_train = pd.read_csv('./Credit/train/overdue_train.txt',names=['ID','Label'],header=None)
bank_detail_train = pd.read_csv('./Credit/train/bank_detail_train.txt',names=['ID','timestamp_bank','type','money','salary'],header=None)
browse_history_train = pd.read_csv('./Credit/train/browse_history_train.txt',names=['ID','timestamp_browse','browse_data','browser_code'],header=None)
bill_detail_train = pd.read_csv('./Credit/train/bill_detail_train.txt',names=['ID','timestamp_bill','bank_id','last_bill','last_repayment','credit_line','current_balance','minimum_payments','amount_transactions','current_money','adjust_money','cycle_interest','available_money','cash_advance','repayment_status'],header=None)

In [5]:
data_user_train = pd.merge(user_info_train,pd.merge(loan_time_train,overdue_train,on='ID',how='inner'),on='ID',how='inner')
data_user_train = data_user_train.apply(lambda x:x.astype(str))

In [6]:
data_user_train.duplicated().sum()

0

In [7]:
data_user_train.isnull().sum()

ID                 0
gender             0
career             0
education          0
marriage           0
hukou              0
timestamp_money    0
Label              0
dtype: int64

In [8]:
pd.crosstab(data_user_train.gender,data_user_train.Label,margins=True)

Label,0,1,All
gender,,,
0,1015,654,1669
1,38616,5638,44254
2,8782,891,9673
All,48413,7183,55596


In [9]:
pd.crosstab(data_user_train.career,data_user_train.Label).apply(lambda x: x/x.sum(), axis=1)

Label,0,1
career,,
0,0.796196,0.203804
1,0.817204,0.182796
2,0.873536,0.126464
3,0.873516,0.126484
4,0.858909,0.141091


In [10]:
pd.crosstab(data_user_train.gender,data_user_train.Label).apply(lambda x: x/x.sum(), axis=1)

Label,0,1
gender,,
0,0.608149,0.391851
1,0.872599,0.127401
2,0.907888,0.092112


In [11]:
pd.crosstab(data_user_train.education,data_user_train.Label).apply(lambda x: x/x.sum(), axis=1)

Label,0,1
education,,
0,0.796748,0.203252
1,0.920354,0.079646
2,0.904161,0.095839
3,0.873481,0.126519
4,0.856447,0.143553


In [12]:
pd.crosstab(data_user_train.marriage,data_user_train.Label).apply(lambda x: x/x.sum(), axis=1)

Label,0,1
marriage,,
0,0.797297,0.202703
1,0.873735,0.126265
2,0.866678,0.133322
3,0.871021,0.128979
4,0.857240,0.142760
5,0.769231,0.230769


In [13]:
pd.crosstab(data_user_train.hukou,data_user_train.Label).apply(lambda x: x/x.sum(), axis=1)

Label,0,1
hukou,,
0,0.796748,0.203252
1,0.881146,0.118854
2,0.862120,0.137880
3,0.880964,0.119036
4,0.863963,0.136037


## Bank information extraction

### 净收入

In [14]:
bank_detail_train['money_direction'] = bank_detail_train['type'].replace({0:1,1:-1})*bank_detail_train['money']
data_net = bank_detail_train.groupby(['ID'])['money_direction'].agg({'sum','count','mean'}).rename(columns=dict(sum='sum_total',count='freq_total',mean='mean_total'))
data_net['ID'] = data_net.index
data_net.reset_index(drop = True)
data_net['ID'] = data_net['ID'].apply(lambda x:str(x))

###  收入信息汇总

In [15]:
data_income = bank_detail_train[bank_detail_train.type==0].groupby(['ID'])['money'].agg({'sum','count','mean'}).rename(columns=dict(sum='sum_income', count='freq_income',mean='mean_income'))
data_income['ID'] = data_income.index
data_income.reset_index(drop = True)
data_income['ID'] = data_income['ID'].apply(lambda x:str(x))

In [16]:
bank_detail_train.groupby("ID")['ID'].count().to_frame().shape

(9294, 1)

In [17]:
len(data_income)/len(data_user_train)

0.16715231311605153

### 支出信息汇总

In [18]:
data_spend = bank_detail_train[bank_detail_train.type==1].groupby(['ID'])['money'].agg({'sum','count','mean'}).rename(columns=dict(sum='sum_spend', count='freq_spend',mean='mean_spend'))
data_spend['ID'] = data_spend.index
data_spend.reset_index(drop = True)
data_spend['ID'] = data_spend['ID'].apply(lambda x:str(x))

In [19]:
data_bank = pd.merge(pd.merge(pd.merge(data_user_train,data_net,on='ID',how='left'),data_income,on='ID',how='left'),data_spend,on='ID',how='left')

In [20]:
data_bank.head()

,ID,gender,career,education,marriage,hukou,timestamp_money,Label,sum_total,mean_total,freq_total,sum_income,mean_income,freq_income,sum_spend,mean_spend,freq_spend
0,3150,1,2,4,1,4,5919867087,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,6965,1,2,4,3,2,5923841487,0,-2261.681747,-6.213411,364.0,972.850228,12.971336,75.0,3234.531975,11.192152,289.0
2,1265,1,3,4,3,1,5915892687,0,-1954.250868,-4.664083,419.0,1708.206195,13.665650,125.0,3662.457063,12.457337,294.0
3,6360,1,2,4,3,2,5923495887,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2583,2,2,2,1,1,5917275087,0,-4327.835360,-5.207985,831.0,2736.475318,12.847302,213.0,7064.310678,11.430923,618.0


## Browser information extraction  

In [21]:
data_browse = browse_history_train.loc[:, ['ID', 'browse_data']].groupby(['ID']).mean()

In [22]:
data_browse.head()

,browse_data
ID,
2,116.091954
3,112.824561
4,106.494505
6,110.000000
7,111.527778


## Bill information extraction

In [23]:
bill_detail_train.columns

Index(['ID', 'timestamp_bill', 'bank_id', 'last_bill', 'last_repayment', 'credit_line', 'current_balance', 'minimum_payments', 'amount_transactions', 'current_money', 'adjust_money', 'cycle_interest', 'available_money', 'cash_advance', 'repayment_status'], dtype='object')

In [24]:
bank_detail_train.columns

Index(['ID', 'timestamp_bank', 'type', 'money', 'salary', 'money_direction'], dtype='object')

In [25]:
bill_detail_train.head()

,ID,timestamp_bill,bank_id,last_bill,last_repayment,credit_line,current_balance,minimum_payments,amount_transactions,current_money,adjust_money,cycle_interest,available_money,cash_advance,repayment_status
0,3150,5906744363,6,18.626118,18.661937,20.664418,18.905766,17.847133,1,0.0,0.0,0.0,0.0,19.971271,0
1,3150,5906744401,6,18.905766,18.909954,20.664418,19.113305,17.911506,1,0.0,0.0,0.0,0.0,19.971271,0
2,3150,5906744427,6,19.113305,19.150290,20.664418,19.300194,17.977610,1,0.0,0.0,0.0,0.0,19.971271,0
3,3150,5906744515,6,19.300194,19.300280,21.000890,20.303240,18.477177,1,0.0,0.0,0.0,0.0,20.307743,0
4,3150,5906744562,6,20.303240,20.307744,21.000890,20.357134,18.510985,1,0.0,0.0,0.0,0.0,20.307743,0


In [26]:
data_bill = bill_detail_train.groupby(['ID'])['credit_line','cash_advance','amount_transactions'].sum()

In [27]:
bill_detail_train.groupby(['ID'])['credit_line','cash_advance','amount_transactions'].sum().head()

,credit_line,cash_advance,amount_transactions
ID,,,
2,428.696405,163.013888,58
3,36.723666,0.000000,4
4,577.987348,281.104005,2
5,41.328836,39.942542,0
6,863.847364,579.114225,122


In [28]:
bill_detail_train.groupby(['ID'])['credit_line','cash_advance','amount_transactions'].sum().reset_index().head()

,ID,credit_line,cash_advance,amount_transactions
0,2,428.696405,163.013888,58
1,3,36.723666,0.000000,4
2,4,577.987348,281.104005,2
3,5,41.328836,39.942542,0
4,6,863.847364,579.114225,122


In [29]:
bill_detail_train.groupby(['ID'])['credit_line','cash_advance','amount_transactions'].sum().head()

,credit_line,cash_advance,amount_transactions
ID,,,
2,428.696405,163.013888,58
3,36.723666,0.000000,4
4,577.987348,281.104005,2
5,41.328836,39.942542,0
6,863.847364,579.114225,122


In [30]:
data_bill = bill_detail_train.assign(bill_credit=bill_detail_train.last_bill/bill_detail_train.credit_line).groupby('ID')['bill_credit'].mean().to_frame().replace({np.inf:2}).reset_index().merge(bill_detail_train.groupby(['ID'])['credit_line','cash_advance','amount_transactions'].sum().reset_index(),on='ID',how='outer')

In [31]:
data_bill.head()

,ID,bill_credit,credit_line,cash_advance,amount_transactions
0,2,2.000000,428.696405,163.013888,58
1,3,1.000269,36.723666,0.000000,4
2,4,0.463136,577.987348,281.104005,2
3,5,1.000717,41.328836,39.942542,0
4,6,0.930503,863.847364,579.114225,122


In [32]:
data_browser = browse_history_train.groupby(['ID','browser_code']).browse_data.sum().reset_index().pivot(index='ID', columns='browser_code', values='browse_data').fillna(0)

In [33]:
data_browser.columns = ['browser'+ str(x) for x in data_browser.columns]

In [34]:
data_browser.reset_index().head()

,ID,browser1,browser2,browser3,browser4,browser5,browser6,browser7,browser8,browser9,browser10,browser11
0,2,59820.0,0.0,11160.0,22725.0,19230.0,4710.0,6150.0,13455.0,0.0,14250.0,0.0
1,3,19968.0,0.0,0.0,7272.0,2694.0,1200.0,2892.0,0.0,0.0,4560.0,0.0
2,4,17824.0,0.0,572.0,9696.0,2188.0,1600.0,1656.0,668.0,0.0,4560.0,0.0
3,6,0.0,0.0,0.0,0.0,0.0,0.0,1650.0,0.0,0.0,0.0,0.0
4,7,53298.0,0.0,0.0,21000.0,10346.0,5740.0,7280.0,2338.0,0.0,12418.0,0.0


## 数据合并

In [35]:
data_bill.ID = data_bill.ID.astype(str)
data_browser = data_browser.reset_index()
data_browser.ID = data_browser.ID.astype(str)

In [44]:
data_train = data_bank.merge(data_bill,on='ID',how='left').merge(data_browser,on='ID',how='left')

In [45]:
data_train.shape

(55596, 32)

In [47]:
data_train.isnull().sum()/len(data_train)

ID                     0.000000
gender                 0.000000
career                 0.000000
education              0.000000
marriage               0.000000
hukou                  0.000000
timestamp_money        0.000000
Label                  0.000000
sum_total              0.832830
mean_total             0.832830
freq_total             0.832830
sum_income             0.832848
mean_income            0.832848
freq_income            0.832848
sum_spend              0.833010
mean_spend             0.833010
freq_spend             0.833010
bill_credit            0.119811
credit_line            0.043564
cash_advance           0.043564
amount_transactions    0.043564
browser1               0.148680
browser2               0.148680
browser3               0.148680
browser4               0.148680
browser5               0.148680
browser6               0.148680
browser7               0.148680
browser8               0.148680
browser9               0.148680
browser10              0.148680
browser1

In [48]:
data_train.shape

(55596, 32)